# Camada GOLD — Agregações Analíticas
Produz tabelas prontas para consumo por dashboards / relatórios.

In [1]:
import sys
sys.path.insert(0, '/opt/spark/work-dir')
from tools.spark_session import get_spark
from pyspark.sql import functions as F

spark = get_spark('Gold - Agregação')
spark.sparkContext.setLogLevel('WARN')

SILVER = '/opt/spark/work-dir/warehouse/silver'
GOLD   = '/opt/spark/work-dir/warehouse/gold'

vendas      = spark.read.format('delta').load(f'{SILVER}/vendas')
clientes    = spark.read.format('delta').load(f'{SILVER}/clientes')
produtos    = spark.read.format('delta').load(f'{SILVER}/produtos')
cupons      = spark.read.format('delta').load(f'{SILVER}/cupons')
avaliacoes  = spark.read.format('delta').load(f'{SILVER}/avaliacoes')

26/06/21 02:54:25 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# Receita mensal
receita_mensal = (
    vendas.groupBy('ano_mes')
    .agg(
        F.count('venda_id').alias('qtd_vendas'),
        F.countDistinct('cliente_id').alias('qtd_clientes_unicos'),
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
        F.round(F.avg('valor_total'), 2).alias('ticket_medio'),
        F.round(F.sum('valor_frete'), 2).alias('receita_frete'),
    )
    .orderBy('ano_mes')
)
receita_mensal.show(15)
receita_mensal.write.format('delta').mode('overwrite').save(f'{GOLD}/receita_mensal')

26/06/21 02:54:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+----------+-------------------+-------------+------------+-------------+
|ano_mes|qtd_vendas|qtd_clientes_unicos|receita_total|ticket_medio|receita_frete|
+-------+----------+-------------------+-------------+------------+-------------+
|2022-01|     11722|              11048|1.491651572E7|     1272.52|    414639.19|
|2022-02|     14916|              13853|1.896282532E7|     1271.31|    525614.21|
|2022-03|     21254|              19148|2.716213626E7|     1277.98|    754279.67|
|2022-04|     18192|              16591| 2.28876322E7|     1258.12|    642275.21|
|2022-05|     23545|              20981| 2.94633595E7|     1251.36|    833605.63|
|2022-06|     15927|              14671| 1.97526383E7|      1240.2|    562866.71|
|2022-07|     16419|              15141| 2.08457498E7|     1269.61|    580457.49|
|2022-08|     19165|              17461|2.410031336E7|     1257.52|    675969.44|
|2022-09|     18223|              16621|2.275930643E7|     1248.93|    639806.55|
|2022-10|     21

In [3]:
# Receita por categoria
receita_categoria = (
    vendas.groupBy('categoria')
    .agg(
        F.count('venda_id').alias('qtd_vendas'),
        F.sum('quantidade').alias('qtd_itens'),
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
        F.round(F.sum('valor_frete'), 2).alias('receita_frete'),
    )
    .orderBy(F.desc('receita_total'))
)
receita_categoria.show()
receita_categoria.write.format('delta').mode('overwrite').save(f'{GOLD}/receita_categoria')

+-----------+----------+---------+--------------+-------------+
|  categoria|qtd_vendas|qtd_itens| receita_total|receita_frete|
+-----------+----------+---------+--------------+-------------+
|Eletrônicos|     73896|   179091|4.0865359438E8|   2615206.86|
| Automotivo|     81017|   196802|1.8108659986E8|   2861345.29|
|   Esportes|     61393|   148600| 5.873174636E7|   2171352.15|
|       Casa|     84243|   205548| 2.931951477E7|   2974603.68|
|     Beleza|     76757|   186788| 2.696784853E7|   2711764.91|
|     Livros|     83865|   204061| 2.215652837E7|   2960458.96|
|     Roupas|     68217|   166592| 1.862771146E7|    2408736.5|
|  Alimentos|     70808|   172289| 1.337117402E7|    2502115.9|
+-----------+----------+---------+--------------+-------------+



In [4]:
# Top 20 clientes por receita
clientes_nome = clientes.select('cliente_id', F.col('nome').alias('nome_cliente'))

top_clientes = (
    vendas.join(clientes_nome, 'cliente_id')
    .groupBy('cliente_id', 'nome_cliente')
    .agg(
        F.count('venda_id').alias('qtd_compras'),
        F.round(F.sum('valor_total'), 2).alias('total_gasto'),
        F.round(F.avg('valor_total'), 2).alias('ticket_medio'),
    )
    .orderBy(F.desc('total_gasto'))
    .limit(20)
)
top_clientes.show()
top_clientes.write.format('delta').mode('overwrite').save(f'{GOLD}/top_clientes')

+----------+--------------------+-----------+-----------+------------+
|cliente_id|        nome_cliente|qtd_compras|total_gasto|ticket_medio|
+----------+--------------------+-----------+-----------+------------+
|     96655|       Rael Siqueira|         11|   81586.86|     7416.99|
|     40876|    Daniela Oliveira|          9|   79968.12|     8885.35|
|      5449|       Rael Nogueira|         11|   79160.03|     7196.37|
|     59386|       Camila Novaes|         10|   78340.27|     7834.03|
|     25545|  Dra. Luara da Mota|          5|   76665.76|    15333.15|
|     69743|         Laís Campos|          8|   74664.19|     9333.02|
|     57416|      Sr. Josué Melo|         11|   72496.24|     6590.57|
|     26655|     Kamilly Freitas|          6|   71669.17|    11944.86|
|      2845|      Hadassa Fogaça|         12|   71210.61|     5934.22|
|     17912|       Bella da Rosa|          7|    71032.5|     10147.5|
|     99114|   Emanuella Machado|          8|   70981.38|     8872.67|
|     

In [5]:
# Receita por região
receita_regiao = (
    vendas.groupBy('regiao')
    .agg(
        F.count('venda_id').alias('qtd_vendas'),
        F.countDistinct('cliente_id').alias('qtd_clientes_unicos'),
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
        F.round(F.avg('valor_total'), 2).alias('ticket_medio'),
    )
    .orderBy(F.desc('receita_total'))
)
receita_regiao.show()
receita_regiao.write.format('delta').mode('overwrite').save(f'{GOLD}/receita_regiao')

+------------+----------+-------------------+--------------+------------+
|      regiao|qtd_vendas|qtd_clientes_unicos| receita_total|ticket_medio|
+------------+----------+-------------------+--------------+------------+
|     Sudeste|    121292|              20126|1.5435866424E8|     1272.62|
|Centro-Oeste|    120925|              20086|1.5268773922E8|     1262.66|
|    Nordeste|    119935|              19946|1.5191350892E8|     1266.63|
|         Sul|    119357|              19862| 1.506304922E8|     1262.02|
|       Norte|    118687|              19708|1.4932431317E8|     1258.14|
+------------+----------+-------------------+--------------+------------+



In [6]:
# Receita por canal
receita_canal = (
    vendas.groupBy('canal')
    .agg(
        F.count('venda_id').alias('qtd_vendas'),
        F.countDistinct('cliente_id').alias('qtd_clientes_unicos'),
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
        F.round(F.avg('valor_total'), 2).alias('ticket_medio'),
        F.round(F.avg('valor_frete'), 2).alias('frete_medio'),
    )
    .orderBy(F.desc('receita_total'))
)
receita_canal.show()
receita_canal.write.format('delta').mode('overwrite').save(f'{GOLD}/receita_canal')

+-----------+----------+-------------------+--------------+------------+-----------+
|      canal|qtd_vendas|qtd_clientes_unicos| receita_total|ticket_medio|frete_medio|
+-----------+----------+-------------------+--------------+------------+-----------+
|     online|    239299|              90845|  3.03404235E8|     1267.89|      35.34|
|loja_fisica|    120557|              69988|1.5349693178E8|     1273.23|      35.31|
|        app|    119758|              69763|1.5190097767E8|      1268.4|      35.34|
| televendas|    120582|              70071| 1.501125733E8|      1244.9|      35.32|
+-----------+----------+-------------------+--------------+------------+-----------+



In [7]:
# Top 20 produtos por receita
top_produtos = (
    vendas.groupBy('produto_id', 'nome', 'categoria')
    .agg(
        F.count('venda_id').alias('qtd_vendas'),
        F.sum('quantidade').alias('qtd_itens'),
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
        F.round(F.avg('valor_total'), 2).alias('ticket_medio'),
    )
    .orderBy(F.desc('receita_total'))
    .limit(20)
)
top_produtos.show()
top_produtos.write.format('delta').mode('overwrite').save(f'{GOLD}/top_produtos')

+----------+--------------+-----------+----------+---------+-------------+------------+
|produto_id|          nome|  categoria|qtd_vendas|qtd_itens|receita_total|ticket_medio|
+----------+--------------+-----------+----------+---------+-------------+------------+
|       931|Smartphone 341|Eletrônicos|       640|     1632|   6934675.81|    10835.43|
|        56|    Tablet 556|Eletrônicos|       581|     1477|   6827095.11|    11750.59|
|       805|Smartphone 909|Eletrônicos|       600|     1515|   6668833.04|    11114.72|
|       701|   Teclado 334|Eletrônicos|       576|     1418|   6525400.52|    11328.82|
|       156|  Notebook 740|Eletrônicos|       570|     1417|   6434911.09|    11289.32|
|       116|    Tablet 907|Eletrônicos|       602|     1464|   6396505.94|    10625.43|
|       944|    Câmera 660|Eletrônicos|       604|     1527|   6335472.05|    10489.19|
|       982|  Smart TV 435|Eletrônicos|       607|     1489|   6322260.73|    10415.59|
|       141|    Câmera 596|Eletr

In [8]:
# Ranking de transportadoras
ranking_transportadoras = (
    vendas.groupBy('transportadora')
    .agg(
        F.count('venda_id').alias('qtd_entregas'),
        F.round(F.avg('valor_frete'), 2).alias('frete_medio'),
        F.round(F.avg('prazo_dias'), 2).alias('prazo_medio_dias'),
        F.round(F.sum('valor_frete'), 2).alias('receita_frete'),
    )
    .orderBy(F.desc('qtd_entregas'))
)
ranking_transportadoras.show()
ranking_transportadoras.write.format('delta').mode('overwrite').save(f'{GOLD}/ranking_transportadoras')

+--------------+------------+-----------+----------------+-------------+
|transportadora|qtd_entregas|frete_medio|prazo_medio_dias|receita_frete|
+--------------+------------+-----------+----------------+-------------+
|      Correios|      150213|      35.35|            4.46|   5309299.71|
|     Braspress|      150172|      35.31|            4.48|   5302952.87|
| Total Express|      149973|      35.37|            4.47|   5303868.28|
|        Jadlog|      149838|       35.3|            4.46|   5289463.39|
+--------------+------------+-----------+----------------+-------------+



In [9]:
# Vendas por turno
vendas_turno = (
    vendas.groupBy('turno')
    .agg(
        F.count('venda_id').alias('qtd_vendas'),
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
        F.round(F.avg('valor_total'), 2).alias('ticket_medio'),
    )
    .orderBy('turno')
)
vendas_turno.show()
vendas_turno.write.format('delta').mode('overwrite').save(f'{GOLD}/vendas_turno')

+-----+----------+--------------+------------+
|turno|qtd_vendas| receita_total|ticket_medio|
+-----+----------+--------------+------------+
|manhã|    211954|2.6834014158E8|     1266.03|
|noite|    176354|2.2287341391E8|     1263.78|
|tarde|    211888|2.6770116226E8|     1263.41|
+-----+----------+--------------+------------+



In [10]:
# Métodos de pagamento
metodos_pagamento = (
    vendas.groupBy('metodo')
    .agg(
        F.count('venda_id').alias('qtd_vendas'),
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
        F.round(F.avg('parcelas'), 2).alias('parcelas_medio'),
        F.round(
            F.count(F.when(F.col('parcelas') > 1, F.lit(1))) * 100.0 / F.count('venda_id'), 2
        ).alias('pct_parcelada'),
    )
    .orderBy(F.desc('receita_total'))
)
metodos_pagamento.show()
metodos_pagamento.write.format('delta').mode('overwrite').save(f'{GOLD}/metodos_pagamento')

+--------------+----------+--------------+--------------+-------------+
|        metodo|qtd_vendas| receita_total|parcelas_medio|pct_parcelada|
+--------------+----------+--------------+--------------+-------------+
|           pix|    240463|3.0416351648E8|           1.0|          0.0|
|cartao_credito|    209980|2.6442345466E8|           6.5|        91.85|
|        boleto|     89704| 1.136041253E8|           1.0|          0.0|
| cartao_debito|     60049| 7.672362131E7|           1.0|          0.0|
+--------------+----------+--------------+--------------+-------------+



In [11]:
# Satisfação por categoria
satisfacao_categoria = (
    avaliacoes.groupBy('categoria')
    .agg(
        F.count('avaliacao_id').alias('qtd_avaliacoes'),
        F.round(F.avg('nota'), 2).alias('nota_media'),
    )
    .orderBy(F.desc('nota_media'))
)
satisfacao_categoria.show()
satisfacao_categoria.write.format('delta').mode('overwrite').save(f'{GOLD}/satisfacao_categoria')

+-----------+--------------+----------+
|  categoria|qtd_avaliacoes|nota_media|
+-----------+--------------+----------+
|     Livros|         58606|      3.61|
|     Beleza|         53754|       3.6|
| Automotivo|         56740|       3.6|
|       Casa|         59002|       3.6|
|  Alimentos|         49728|       3.6|
|     Roupas|         47628|       3.6|
|Eletrônicos|         51802|      3.59|
|   Esportes|         42877|      3.59|
+-----------+--------------+----------+



In [12]:
# Satisfação por região
satisfacao_regiao = (
    avaliacoes
    .join(clientes.select('cliente_id', 'regiao'), 'cliente_id')
    .groupBy('regiao')
    .agg(
        F.count('avaliacao_id').alias('qtd_avaliacoes'),
        F.round(F.avg('nota'), 2).alias('nota_media'),
    )
    .orderBy(F.desc('nota_media'))
)
satisfacao_regiao.show()
satisfacao_regiao.write.format('delta').mode('overwrite').save(f'{GOLD}/satisfacao_regiao')

+------------+--------------+----------+
|      regiao|qtd_avaliacoes|nota_media|
+------------+--------------+----------+
|    Nordeste|         84064|       3.6|
|         Sul|         83601|       3.6|
|     Sudeste|         84918|       3.6|
|Centro-Oeste|         84580|       3.6|
|       Norte|         82974|       3.6|
+------------+--------------+----------+



In [13]:
# Estoque crítico — produtos sem estoque
estoque_critico = (
    produtos
    .filter(F.col('disponivel') == False)
    .select('produto_id', 'nome', 'categoria', 'estoque', 'razao_social')
    .orderBy('estoque')
)
estoque_critico.show()
estoque_critico.write.format('delta').mode('overwrite').save(f'{GOLD}/estoque_critico')

+----------+------------+-----------+-------+--------------------+
|produto_id|        nome|  categoria|estoque|        razao_social|
+----------+------------+-----------+-------+--------------------+
|       119|Python e 722|     Livros|      0|              Guerra|
|       175|  Blazer 717|     Roupas|      0|        Fogaça Ltda.|
|       263| Arte de 870|     Livros|      0|         Farias S.A.|
|       684|   Mouse 628|Eletrônicos|      0|Aparecida das Nev...|
+----------+------------+-----------+-------+--------------------+



In [14]:
# Segmentação RFM dos clientes
clientes_rfm = (
    vendas
    .groupBy('cliente_id')
    .agg(
        F.datediff(F.current_date(), F.max('data_venda')).alias('recencia_dias'),
        F.count('venda_id').alias('frequencia'),
        F.round(F.sum('valor_total'), 2).alias('monetario'),
    )
    .join(clientes.select('cliente_id', 'nome', 'regiao', 'faixa_score'), 'cliente_id')
    .orderBy(F.desc('monetario'))
)
clientes_rfm.show(10)
clientes_rfm.write.format('delta').mode('overwrite').save(f'{GOLD}/clientes_rfm')

+----------+-------------+----------+---------+------------------+------------+-----------+
|cliente_id|recencia_dias|frequencia|monetario|              nome|      regiao|faixa_score|
+----------+-------------+----------+---------+------------------+------------+-----------+
|     96655|          825|        11| 81586.86|     Rael Siqueira|    Nordeste|        bom|
|     40876|          798|         9| 79968.12|  Daniela Oliveira|         Sul|       ruim|
|      5449|          883|        11| 79160.03|     Rael Nogueira|     Sudeste|    regular|
|     59386|          738|        10| 78340.27|     Camila Novaes|         Sul|       ruim|
|     25545|          851|         5| 76665.76|Dra. Luara da Mota|Centro-Oeste|    regular|
|     69743|          825|         8| 74664.19|       Laís Campos|Centro-Oeste|        bom|
|     57416|          800|        11| 72496.24|    Sr. Josué Melo|       Norte|        bom|
|     26655|         1174|         6| 71669.17|   Kamilly Freitas|       Norte| 

In [15]:
# Performance de cupons
cupons_performance = (
    vendas
    .filter(F.col('cupom_id').isNotNull())
    .groupBy('cupom_id')
    .agg(
        F.count('venda_id').alias('qtd_usos'),
        F.round(F.sum('valor_total'), 2).alias('receita_com_cupom'),
        F.round(F.avg('valor_total'), 2).alias('ticket_medio'),
    )
    .join(cupons.select('cupom_id', 'codigo', 'tipo', 'valor', 'categoria'), 'cupom_id')
    .orderBy(F.desc('qtd_usos'))
)
cupons_performance.show()
cupons_performance.write.format('delta').mode('overwrite').save(f'{GOLD}/cupons_performance')

+--------+--------+-----------------+------------+--------+----------+-----+-----------+
|cupom_id|qtd_usos|receita_com_cupom|ticket_medio|  codigo|      tipo|valor|  categoria|
+--------+--------+-----------------+------------+--------+----------+-----+-----------+
|      29|    3549|       4171234.06|     1175.33| AUT8872|percentual| 17.0|   Esportes|
|      96|    3491|       4373695.44|     1252.85|LABO8819|percentual| 11.0|Eletrônicos|
|      44|    1832|        2134844.3|     1165.31|ALIQ4222|percentual| 12.0|Eletrônicos|
|      32|    1786|       2325511.48|     1302.08| SIT9800|percentual| 12.0|      TODAS|
|      68|    1253|       1471944.99|     1174.74|ODIT8619|percentual|  5.0|       Casa|
|      64|    1180|       1305108.91|     1106.02|IURE4902|percentual| 22.0|Eletrônicos|
|     164|    1176|       1536513.59|     1306.56|QUIB8894|      fixo|35.72|  Alimentos|
|      40|    1173|       1657574.06|     1413.11|IPSA8913|      fixo|14.83|     Livros|
|     111|    1139|  

In [16]:
# Histórico de versões de cada tabela Gold
from delta.tables import DeltaTable
gold_tables = [
    'receita_mensal', 'receita_categoria', 'receita_regiao', 'receita_canal',
    'top_produtos', 'top_clientes', 'ranking_transportadoras', 'vendas_turno',
    'metodos_pagamento', 'satisfacao_categoria', 'satisfacao_regiao',
    'estoque_critico', 'clientes_rfm', 'cupons_performance',
]
for name in gold_tables:
    dt = DeltaTable.forPath(spark, f'{GOLD}/{name}')
    v  = dt.history(1).collect()[0]
    print(f'[gold/{name}] v{v["version"]} — {v["operation"]}')

[gold/receita_mensal] v0 — WRITE
[gold/receita_categoria] v0 — WRITE
[gold/receita_regiao] v0 — WRITE
[gold/receita_canal] v0 — WRITE
[gold/top_produtos] v0 — WRITE
[gold/top_clientes] v0 — WRITE
[gold/ranking_transportadoras] v0 — WRITE
[gold/vendas_turno] v0 — WRITE
[gold/metodos_pagamento] v0 — WRITE
[gold/satisfacao_categoria] v0 — WRITE
[gold/satisfacao_regiao] v0 — WRITE
[gold/estoque_critico] v0 — WRITE
[gold/clientes_rfm] v0 — WRITE
[gold/cupons_performance] v0 — WRITE
